# Thin Bridges — Results Visualisation

Reproduces all figures from the NeurIPS 2025 paper *"Thin Bridges for Drug Text Alignment"*.

**Two modes:**
-  → reproduces Table 1 row 4: Recall@1=0.762, MRR=0.863 (run )
-  → reproduces Table 1 row 5: Recall@1=0.150, MRR=0.228 (run )

**Prerequisites:**
```bash
python 01_download_data.py
python 03b_train_global_bridge.py   --csv chembl_mechanisms_enriched.csv  # for global
python 03_train_scaffold_split.py   --csv chembl_mechanisms_enriched.csv  # for scaffold
```

In [ ]:
import os, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, matplotlib.ticker as mtick
import seaborn as sns

# ── SET THIS to switch between the two result settings ──────────────────
MODE   = "global"    # "global" → Recall@1=0.762 | "scaffold" → Recall@1=0.150
OUTDIR = "outputs"
# ────────────────────────────────────────────────────────────────────────

sns.set_theme(style="whitegrid", font_scale=1.1)
print(f"Mode: {MODE}")

## 1. Load test embeddings and model

In [ ]:
prefix = "global_" if MODE == "global" else ""

Z_text = torch.load(os.path.join(OUTDIR, f"{prefix}Z_text.pt"),  map_location="cpu")
X_mol  = np.load(   os.path.join(OUTDIR, f"{prefix}X_mol.npy"))
df     = pd.read_csv(os.path.join(OUTDIR, f"{prefix}df.csv"))
epoch_losses = np.load(os.path.join(OUTDIR, f"{prefix}epoch_losses.npy"))

SHARED_D = 256
d_text   = Z_text.shape[1]
d_fp     = X_mol.shape[1]
device   = "cuda" if torch.cuda.is_available() else "cpu"

proj_text   = nn.Linear(d_text, SHARED_D).to(device)
proj_mol    = nn.Linear(d_fp,   SHARED_D).to(device)
proj_text_0 = nn.Linear(d_text, SHARED_D).to(device)
proj_mol_0  = nn.Linear(d_fp,   SHARED_D).to(device)

proj_text.load_state_dict(  torch.load(os.path.join(OUTDIR, f"{prefix}proj_text.pt"),   map_location=device))
proj_mol.load_state_dict(   torch.load(os.path.join(OUTDIR, f"{prefix}proj_mol.pt"),    map_location=device))
proj_text_0.load_state_dict(torch.load(os.path.join(OUTDIR, f"{prefix}proj_text_0.pt"), map_location=device))
proj_mol_0.load_state_dict( torch.load(os.path.join(OUTDIR, f"{prefix}proj_mol_0.pt"),  map_location=device))

print(f"Loaded {MODE} artefacts | {len(df)} pairs | d_text={d_text} | d_fp={d_fp}")

## 2. Helper functions

In [ ]:
@torch.no_grad()
def similarity_from_heads(Zt, Xm, pt, pm, device='cpu'):
    """Project frozen embeddings through heads and compute cosine similarity."""
    Bt = F.normalize(pt(torch.tensor(Zt, dtype=torch.float32).to(device)), dim=1).cpu()
    Bm = F.normalize(pm(torch.tensor(Xm, dtype=torch.float32).to(device)), dim=1).cpu()
    return (Bt @ Bm.T)   # [N, N]

def recall_mrr(S_np):
    N = S_np.shape[0]
    gt   = np.arange(N)
    top1 = S_np.argmax(axis=1)
    recall1 = (top1 == gt).mean()
    ranks = [1.0 / (int(np.where(np.argsort(-S_np[i]) == i)[0][0]) + 1) for i in range(N)]
    return float(recall1), float(np.mean(ranks))

def grouped_recall_at1(S_np, meta_df, group_col='target_chembl_id', min_group=3):
    groups = meta_df[group_col].fillna('UNK').astype(str).values
    idx_by_group = {}
    for i, g in enumerate(groups):
        idx_by_group.setdefault(g, []).append(i)
    hits = total = 0
    for g, idxs in idx_by_group.items():
        if len(idxs) < min_group:
            continue
        sub  = S_np[np.ix_(idxs, idxs)]
        top1 = sub.argmax(axis=1)
        hits  += (top1 == np.arange(len(idxs))).sum()
        total += len(idxs)
    return hits / total if total else 0.0

def recall_at_k(S_np, k_list=(1, 2, 3, 4, 5, 10)):
    N = S_np.shape[0]
    topk_all = np.argsort(-S_np, axis=1)
    return {k: sum(i in topk_all[i, :k] for i in range(N)) / N for k in k_list}

def within_target_recall_at_k(S_np, meta_df, group_col='target_chembl_id',
                               min_group=3, k_list=(1, 2, 3, 4, 5, 10)):
    groups = meta_df[group_col].fillna('UNK').astype(str).values
    idx_by_group = {}
    for i, g in enumerate(groups):
        idx_by_group.setdefault(g, []).append(i)
    results = {k: 0 for k in k_list}
    total = 0
    for g, idxs in idx_by_group.items():
        if len(idxs) < min_group:
            continue
        sub = S_np[np.ix_(idxs, idxs)]
        topk_all = np.argsort(-sub, axis=1)
        for ki, k in enumerate(k_list):
            results[k] += sum(j in topk_all[j, :k] for j in range(len(idxs)))
        total += len(idxs)
    return {k: v / total if total else 0.0 for k, v in results.items()}

# Compute similarity matrices
S_before = similarity_from_heads(Z_text_test.numpy(), X_mol_test,
                                  proj_text_0, proj_mol_0, device=device)
S_after  = similarity_from_heads(Z_text_test.numpy(), X_mol_test,
                                  proj_text,   proj_mol,   device=device)
S_before_np = S_before.numpy()
S_after_np  = S_after.numpy()

# Pre-trained (direct ECFP4 @ text CLS, no heads) baseline
E_text_n   = F.normalize(Z_text_test, dim=1)
E_mol_n    = F.normalize(torch.tensor(X_mol_test), dim=1)
S_frozen_np = (E_text_n @ E_mol_n.T).numpy()

print('Similarity matrices ready.')

## 3. Summary metrics table (replicates Table 1)

In [ ]:
rec1_frozen, mrr_frozen = recall_mrr(S_frozen_np)
rec1_before, mrr_before = recall_mrr(S_before_np)
rec1_after,  mrr_after  = recall_mrr(S_after_np)
gR1_after = grouped_recall_at1(S_after_np, test_df)

results = pd.DataFrame([
    {'Setting': 'Frozen encoders (ECFP4 + S-Biomed, no bridge)', 'Recall@1': rec1_frozen, 'MRR': mrr_frozen, 'Grouped R@1': '-'},
    {'Setting': 'Random projection heads (before training)',      'Recall@1': rec1_before, 'MRR': mrr_before, 'Grouped R@1': '-'},
    {'Setting': 'Scaffold split (ECFP4 + text_rich bridge)',      'Recall@1': rec1_after,  'MRR': mrr_after,  'Grouped R@1': f'{gR1_after:.3f}'},
])
results['Recall@1'] = results['Recall@1'].apply(lambda x: f'{x:.3f}' if isinstance(x, float) else x)
results['MRR']      = results['MRR'].apply(     lambda x: f'{x:.3f}' if isinstance(x, float) else x)
results.set_index('Setting', inplace=True)
display(results)

## 4. Figure 1 / Figure 2 — Before vs After cosine similarity heatmaps

In [ ]:
K = min(40, S_after_np.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(S_before_np[:K, :K], vmin=-1, vmax=1, cmap='viridis', ax=axes[0])
axes[0].set_title('BEFORE (random heads) — TEST split')
axes[0].set_xlabel('Molecule index'); axes[0].set_ylabel('Text index')

sns.heatmap(S_after_np[:K, :K], vmin=-1, vmax=1, cmap='viridis', ax=axes[1])
axes[1].set_title('AFTER (trained heads) — TEST split')
axes[1].set_xlabel('Molecule index'); axes[1].set_ylabel('Text index')

plt.suptitle('ECFP4 bridge with enriched text (text_rich)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig_before_after_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Figure 3 — Cumulative Match (Recall@k) curve

In [ ]:
k_list = [1, 2, 3, 4, 5, 10]

global_rk       = recall_at_k(S_after_np, k_list=k_list)
within_target_rk = within_target_recall_at_k(S_after_np, test_df,
                                              min_group=3, k_list=k_list)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(k_list, [global_rk[k]        for k in k_list], 'o-', label='Global',        linewidth=2)
ax.plot(k_list, [within_target_rk[k] for k in k_list], 's-', label='Within-target', linewidth=2)
ax.set_xlabel('k'); ax.set_ylabel('Recall@k')
ax.set_title('Cumulative Match (Recall@k) — Scaffold Split Test')
ax.legend(); ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f'{y:.2f}'))
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig_recall_at_k.png'), dpi=150)
plt.show()

print('Global Recall@k:')
for k, v in global_rk.items(): print(f'  @{k}: {v:.3f}')
print('Within-target Recall@k:')
for k, v in within_target_rk.items(): print(f'  @{k}: {v:.3f}')

## 6. Training loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(epoch_losses)+1), epoch_losses, linewidth=1.5, color='steelblue')
ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss')
ax.set_title('Contrastive Bridge Training Loss')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig_training_loss.png'), dpi=150)
plt.show()

## 7. Figure 4 — Ablation: Grouped Recall@1 by setting

In [ ]:
# This cell runs ablation over temperature T and margin m using the already-trained
# model embeddings. Note: a true ablation requires retraining for each setting.
# Here we approximate the effect by rescaling the logits (T only).
# To reproduce exact Figure 4, rerun 03_train_scaffold_split.py with different TEMP/MARGIN.

settings = [
    ('WithDrug | T=0.05 | m=0.15', 0.05),
    ('WithDrug | T=0.07 | m=0.05', 0.07),
    ('WithDrug | T=0.05 | m=0.05', 0.05),
    ('WithDrug | T=0.05 | m=0.10', 0.05),
    ('WithDrug | T=0.07 | m=0.10', 0.07),
    ('WithDrug | T=0.10 | m=0.15', 0.10),
    ('WithDrug | T=0.10 | m=0.05', 0.10),
    ('WithDrug | T=0.07 | m=0.15', 0.07),
]

# Approximate by re-scaling the AFTER similarity matrix
grp_scores = []
for label, T in settings:
    S_scaled = S_after_np / T   # approximate temperature scaling
    gR1 = grouped_recall_at1(S_scaled, test_df, min_group=3)
    grp_scores.append((label, gR1))

grp_scores.sort(key=lambda x: -x[1])
labels, scores = zip(*grp_scores)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(labels)), scores, color='steelblue', alpha=0.85)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Grouped Recall@1')
ax.set_title('Ablation: Grouped R@1 by setting (approximated)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig_ablation.png'), dpi=150)
plt.show()

## 8. Top-1 retrieval examples (correct and wrong)

In [ ]:
def show_top1_examples(S_np, meta_df, num=5, text_col='text_rich'):
    N    = S_np.shape[0]
    gt   = np.arange(N)
    top1 = S_np.argmax(axis=1)
    col  = text_col if text_col in meta_df.columns else 'mechanism_of_action'

    correct = np.where(top1 == gt)[0]
    wrong   = np.where(top1 != gt)[0]

    print('--- Correct Top-1 matches (TEST) ---')
    for idx in correct[:num]:
        j = top1[idx]
        print(f'[i={idx}] sim={S_np[idx, j]:.3f}')
        print(f'  TEXT:  {meta_df.iloc[idx][col]}')
        print(f'  SMILES:{meta_df.iloc[j]["smiles"]}')
        print()

    print('--- Wrong Top-1 matches (TEST) ---')
    for idx in wrong[:num]:
        j = top1[idx]
        print(f'[i={idx}] sim(best)={S_np[idx,j]:.3f}  (true sim={S_np[idx,idx]:.3f})')
        print(f'  TEXT:        {meta_df.iloc[idx][col]}')
        print(f'  SMILES_best: {meta_df.iloc[j]["smiles"]}')
        print(f'  SMILES_true: {meta_df.iloc[idx]["smiles"]}')
        print()

show_top1_examples(S_after_np, test_df, num=3)

## 9. Diagonal score distribution

In [ ]:
pos_scores_before = np.diag(S_before_np)
neg_scores_before = S_before_np[~np.eye(S_before_np.shape[0], dtype=bool)]

pos_scores_after  = np.diag(S_after_np)
neg_scores_after  = S_after_np[~np.eye(S_after_np.shape[0], dtype=bool)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

axes[0].hist(neg_scores_before, bins=60, alpha=0.6, label='Negatives', density=True, color='steelblue')
axes[0].hist(pos_scores_before, bins=30, alpha=0.8, label='Positives', density=True, color='orange')
axes[0].set_title('Before training'); axes[0].set_xlabel('Cosine similarity'); axes[0].legend()

axes[1].hist(neg_scores_after, bins=60, alpha=0.6, label='Negatives', density=True, color='steelblue')
axes[1].hist(pos_scores_after, bins=30, alpha=0.8, label='Positives', density=True, color='orange')
axes[1].set_title('After training'); axes[1].set_xlabel('Cosine similarity'); axes[1].legend()

plt.suptitle('Positive vs Negative similarity score distribution', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'BEFORE — pos mean: {pos_scores_before.mean():.3f}  neg mean: {neg_scores_before.mean():.3f}')
print(f'AFTER  — pos mean: {pos_scores_after.mean():.3f}  neg mean: {neg_scores_after.mean():.3f}')

## 10. Per-target group size vs Recall@1 scatter

In [ ]:
groups = test_df['target_chembl_id'].fillna('UNK').astype(str).values
idx_by_group = {}
for i, g in enumerate(groups):
    idx_by_group.setdefault(g, []).append(i)

group_sizes, group_recalls = [], []
for g, idxs in idx_by_group.items():
    if len(idxs) < 2:
        continue
    sub  = S_after_np[np.ix_(idxs, idxs)]
    top1 = sub.argmax(axis=1)
    r1   = (top1 == np.arange(len(idxs))).mean()
    group_sizes.append(len(idxs))
    group_recalls.append(r1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(group_sizes, group_recalls, alpha=0.6, edgecolors='k', linewidths=0.3, s=40)
ax.set_xlabel('Group size (# compounds per target)')
ax.set_ylabel('Within-target Recall@1')
ax.set_title('Per-target retrieval performance vs group size')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig_group_scatter.png'), dpi=150)
plt.show()